# 🎯 02 - Center and Spread

Module 06 used `.describe()` as a first look, to spot problems. Now we read it properly. Every numeric column can be summarized with two questions:

1. **Where is the middle?** (center: mean, median, mode)
2. **How spread out are the values?** (spread: range, standard deviation, IQR, percentiles)

Two columns can have the same center and completely different spreads, and a single number is never the whole story.

## ✅ Learning Goals
- Compute the mean, median, and mode with pandas methods, and explain when they disagree
- Show that the median is **robust** to outliers and the mean is not
- Compute range, variance, standard deviation, IQR, and any percentile
- Explain `ddof`, and why pandas and NumPy give different standard deviations by default
- Read every row of `.describe()`, including what `count` tells you about missing data
- See how filling missing values (Module 06) changes the statistics

In [1]:
import numpy as np
import pandas as pd

titanic = pd.read_csv('titanic.csv')
fare = titanic['Fare']
age = titanic['Age']

## 1. Center: Mean, Median, Mode
Every pandas `Series` has these built in, so no NumPy or SciPy is needed:

| Statistic | Method | Meaning |
|---|---|---|
| Mean | `.mean()` | The sum divided by the count. The "balance point." |
| Median | `.median()` | The middle value after sorting. Half the values are below it. |
| Mode | `.mode()` | The most common value (or values). |

In [2]:
print(f'Fare mean:   {fare.mean():.2f}')
print(f'Fare median: {fare.median():.2f}')
print('Fare mode:  ', fare.mode().tolist())

Fare mean:   32.20
Fare median: 14.45
Fare mode:   [8.05]


The mean fare is more than **twice** the median. That gap is a signal, and the next section shows what causes it.

`.mode()` returns a **Series**, not a single number, because a column can have a tie for most common value. Use `.mode()[0]` when you need just one.

## 2. Robustness: Who Gets Pulled by Outliers?
A statistic is **robust** if a few extreme values barely move it. Watch what happens to a small, made-up set of fares when one very expensive ticket is added:

In [3]:
fares = pd.Series([8, 10, 12, 15, 20])
with_outlier = pd.Series([8, 10, 12, 15, 20, 500])

print('Without outlier: mean', fares.mean(), '| median', fares.median())
print('With outlier:    mean', round(with_outlier.mean(), 1), '| median', with_outlier.median())

Without outlier: mean 13.0 | median 12.0
With outlier:    mean 94.2 | median 13.5


One value moved the mean from 13 to about 94. The median barely changed (12 → 13.5), because it only cares about the **middle position**, not how far away the extreme values are.

That's what is happening in the real Titanic fares: a small number of very expensive 1st-class tickets pull the mean far above what a typical passenger paid.

| Use the… | When… |
|---|---|
| **Median** | the data are lopsided or have extreme values (incomes, prices, house sizes) |
| **Mean** | the data are roughly symmetric, or you need a total (mean × count = total) |
| **Mode** | the data are categories, or a few repeated values (most common shoe size) |

✅ **Your Turn:** Compute the mean and median of `age`. Are they close? What does that suggest about whether age has extreme values?

In [5]:
# Your turn here
print(f'Age mean:   {titanic["Age"].mean():.2f}')
print(f'Age median: {titanic["Age"].median():.2f}')
print('Age mode:  ', titanic["Age"].mode().tolist())
# No extreme values in Age column, so mean and median are similar.

Age mean:   29.70
Age median: 28.00
Age mode:   [24.0]


## 3. Spread: Range, Variance, Standard Deviation
- **Range** = max − min. Easy, but it depends on only the two most extreme values.
- **Variance** = the average squared distance from the mean. Its units are squared (pounds²), which is hard to interpret.
- **Standard deviation (std)** = the square root of the variance, back in the original units. Roughly: how far a typical value is from the mean.

In [6]:
print(f'Age range: {age.max() - age.min():.2f} years ({age.min()} to {age.max()})')
print(f'Age variance: {age.var():.2f} (years squared)')
print(f'Age std: {age.std():.2f} years')

Age range: 79.58 years (0.42 to 80.0)
Age variance: 211.02 (years squared)
Age std: 14.53 years


### `ddof`: Why Pandas and NumPy Disagree
Compute the standard deviation of the same column two ways:

In [7]:
known_ages = age.dropna()

print('pandas .std():', known_ages.std())
print('np.std():     ', np.std(known_ages))

pandas .std(): 14.526497332334042
np.std():      14.516321150817316


They're different, and neither is a bug. The difference is the **`ddof`** ("delta degrees of freedom") setting:

| | Default | Divides by | Use when |
|---|---|---|---|
| pandas `.std()` / `.var()` | `ddof=1` | n − 1 | your data are a **sample** of a bigger population |
| NumPy `np.std()` / `np.var()` | `ddof=0` | n | your data **are** the whole population |

Dividing by n − 1 corrects for the fact that a sample tends to look a little less spread out than the population it came from. Titanic's 891 rows are a sample of the roughly 2,200 people aboard, so pandas' default is the right one here. With large n the difference is tiny. With 5 values, it isn't.

You can always set it explicitly, and the two will match:

In [8]:
print(known_ages.std(ddof=0))
print(np.std(known_ages, ddof=0))

14.516321150817316
14.516321150817316


## 4. Percentiles, Quartiles, and the IQR
A **percentile** answers "what value is this percentage of the data below?" The 90th percentile of fare is the price that 90% of passengers paid less than.

The **quartiles** are the 25th, 50th, and 75th percentiles (Q1, the median, and Q3). The **interquartile range (IQR)** is Q3 − Q1: the width of the **middle half** of the data. Like the median, it ignores the extremes, so it's robust.

In [9]:
fare.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.99])

0.10      7.55000
0.25      7.91040
0.50     14.45420
0.75     31.00000
0.90     77.95830
0.99    249.00622
Name: Fare, dtype: float64

In [10]:
q1 = fare.quantile(0.25)
q3 = fare.quantile(0.75)
iqr = q3 - q1

print(f'Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}')
print(f'Fare std = {fare.std():.2f}')

Q1 = 7.91, Q3 = 31.00, IQR = 23.09
Fare std = 49.69


The middle half of passengers paid within a band about 23 wide, but the standard deviation is about 50. The std is inflated by the same expensive tickets that pulled the mean up. **Robust center (median) goes with robust spread (IQR); mean goes with std.**

Notice the jump from the 90th percentile (about 78) to the 99th (about 250). The top 1% of fares are in a different world from everyone else. Notebook 03 takes this apart.

## 5. Reading `.describe()` Row by Row
You've run `.describe()` since Module 06. Now you know what every row means:

In [11]:
titanic[['Age', 'Fare', 'SibSp']].describe()

,Age,Fare,SibSp
count,714.000000,891.000000,891.000000
mean,29.699118,32.204208,0.523008
std,14.526497,49.693429,1.102743
min,0.420000,0.000000,0.000000
25%,20.125000,7.910400,0.000000
50%,28.000000,14.454200,0.000000
75%,38.000000,31.000000,1.000000
max,80.000000,512.329200,8.000000


| Row | Meaning | What to look for |
|---|---|---|
| `count` | Non-missing values | Lower than the number of rows? Missing data. `Age` has 714 of 891. |
| `mean` / `50%` | Mean and median | Far apart? Lopsided data or outliers (`Fare`). |
| `std` | Standard deviation | Large relative to the mean? Very spread out. |
| `min` / `max` | The extremes | Impossible values? (A negative age, a $0 fare.) |
| `25%` / `75%` | Q1 and Q3 | Their difference is the IQR. |

You can ask for different percentiles, and turn the table sideways (`.T`) when there are many columns:

In [12]:
titanic[['Age', 'Fare']].describe(percentiles=[0.05, 0.5, 0.95]).T.round(2)

,count,mean,std,min,5%,50%,95%,max
Age,714.0,29.7,14.53,0.42,4.00,28.00,56.00,80.00
Fare,891.0,32.2,49.69,0.00,7.22,14.45,112.08,512.33


### 🧠 Quick Check
`Fare` has a `min` of `0.00`. Is a $0 fare a data error, a placeholder (Module 06), or something real? How could you investigate? (Hint: filter for those rows and look at the other columns.)

## 6. Summarizing Categorical Columns
`.describe()` on text columns gives a different summary: count, number of distinct values, the most common value (`top`), and how often it appears (`freq`).

In [13]:
titanic[['Sex', 'Embarked']].describe()

,Sex,Embarked
count,891,889
unique,2,3
top,male,S
freq,577,644


In [14]:
# The proportion in each category: the categorical version of a "summary"
titanic['Embarked'].value_counts(normalize=True, dropna=False).round(3)

Embarked
S      0.723
C      0.189
Q      0.086
NaN    0.002
Name: proportion, dtype: float64

## 7. Missing Values Change the Statistics
Pandas statistics **skip** missing values by default (`skipna=True`). So `age.mean()` is the mean of the **714 known ages**, not all 891 passengers.

In Module 06 you learned to fill missing values. Watch what a "harmless" median fill does to the spread:

In [15]:
age_filled = age.fillna(age.median())

comparison = pd.DataFrame({
    'original': age.describe(),
    'median-filled': age_filled.describe(),
}).round(2)
comparison

,original,median-filled
count,714.00,891.00
mean,29.70,29.36
std,14.53,13.02
min,0.42,0.42
25%,20.12,22.00
50%,28.00,28.00
75%,38.00,35.00
max,80.00,80.00


The median didn't move, but the **std shrank** and the quartiles squeezed toward the middle. You added 177 identical values right at the center, so the data look *less* spread out than they really are.

> ⚠️ Filling missing values is a decision with consequences. If you fill, say so, and compare the statistics before and after, the way the Module 06 lab compared raw and cleaned data.

---
### 📝 Summary
- Center: `.mean()`, `.median()`, `.mode()`. The median is robust to outliers, and the mean is not.
- Spread: range, `.var()`, `.std()`, `.quantile()`, and IQR = Q3 − Q1. Pair median with IQR, and mean with std.
- Pandas uses `ddof=1` (sample) and NumPy uses `ddof=0` (population). Set it explicitly when it matters.
- `.describe()`: `count` reveals missing data, and a big mean-vs-median gap reveals lopsided data.
- Statistics skip `NaN` by default, and filling missing values can quietly shrink the spread.

## 🏋️ Practice

### Practice 1 — Center of Pokémon Stats (easy)
Load `pokemon.csv` into `poke`. For `HP`, `Attack`, and `Speed`, print the mean and median. Which column has the biggest gap between them?

In [18]:
poke = pd.read_csv('pokemon.csv')
# Your code here
HP = poke['HP']
Attack = poke['Attack']
Speed = poke['Speed']
print(f'HP mean:   {HP.mean():.2f}', f'HP median: {HP.median():.2f}', f'HP gap: HP mean - HP median = {HP.mean() - HP.median():.2f}')
print(f'Attack mean:   {Attack.mean():.2f}', f'Attack median: {Attack.median():.2f}', f'Attack gap: Attack mean - Attack median = {Attack.mean() - Attack.median():.2f}')
print(f'Speed mean:   {Speed.mean():.2f}', f'Speed median: {Speed.median():.2f}', f'Speed gap: Speed mean - Speed median = {Speed.mean() - Speed.median():.2f}')
print(f'HP has the largest gap between mean and median, indicating that it is the most skewed distribution among the three variables.')

HP mean:   69.26 HP median: 65.00 HP gap: HP mean - HP median = 4.26
Attack mean:   79.00 Attack median: 75.00 Attack gap: Attack mean - Attack median = 4.00
Speed mean:   68.28 Speed median: 65.00 Speed gap: Speed mean - Speed median = 3.28
HP has the largest gap between mean and median, indicating that it is the most skewed distribution among the three variables.


### Practice 2 — Same Mean, Different Spread (medium)
Create two Series with **the same mean** (for example, both average to 50) but very different standard deviations. Print the mean, std, and IQR of each. In a comment, describe how a group of people with each set of scores would be different.

In [19]:
# Your code here

# Same mean (50), very different spread

scores_consistent = pd.Series([48, 49, 50, 51, 52])
scores_variable = pd.Series([0, 25, 50, 75, 100])

for name, s in {
    "Consistent": scores_consistent,
    "Variable": scores_variable
}.items():
    
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    
    print(f"\n{name}")
    print("Mean:", s.mean())
    print("Std :", round(s.std(), 2))
    print("IQR :", iqr)


Consistent
Mean: 50.0
Std : 1.58
IQR : 2.0

Variable
Mean: 50.0
Std : 39.53
IQR : 50.0


### Comment: 
Both groups have the same average score (50), but they are very different. The "Consistent" group is clustered tightly around 50, meaning most people performed similarly. The standard deviation and IQR are small.
The "Variable" group contains both very low and very high scores. Some people performed poorly while others performed extremely well. The average is still 50, but the large standard deviation and IQR reveal much greater variation in performance.

### Practice 3 — Zero Fares (harder)
Filter `titanic` for rows where `Fare == 0`. How many are there? Look at their `Pclass`, `Age`, `Ticket`, and `Embarked` columns. Then compute the mean and median `Fare` with and without them. In a Markdown cell, say whether you'd treat them as placeholders, and why.

In [20]:
# Your code here
# Filter passengers with Fare == 0
zero_fare = titanic[titanic['Fare'] == 0]

print("Number of passengers with Fare == 0:")
print(len(zero_fare))

# Examine relevant columns
zero_fare[['Pclass', 'Age', 'Ticket', 'Embarked']]

Number of passengers with Fare == 0:
15


,Pclass,Age,Ticket,Embarked
179,3,36.0,LINE,S
263,1,40.0,112059,S
271,3,25.0,LINE,S
277,2,NaN,239853,S
302,3,19.0,LINE,S
413,2,NaN,239853,S
466,2,NaN,239853,S
481,2,NaN,239854,S
597,3,49.0,LINE,S
633,1,NaN,112052,S


In [21]:
# Including zero fares
mean_with = titanic['Fare'].mean()
median_with = titanic['Fare'].median()

# Excluding zero fares
nonzero = titanic[titanic['Fare'] > 0]

mean_without = nonzero['Fare'].mean()
median_without = nonzero['Fare'].median()

print("Including zeros")
print("Mean:", mean_with)
print("Median:", median_with)

print("\nExcluding zeros")
print("Mean:", mean_without)
print("Median:", median_without)

Including zeros
Mean: 32.204207968574636
Median: 14.4542

Excluding zeros
Mean: 32.75564988584475
Median: 14.5


In [22]:
pd.DataFrame({
    'Including Fare=0': [mean_with, median_with],
    'Excluding Fare=0': [mean_without, median_without]
}, index=['Mean', 'Median'])

,Including Fare=0,Excluding Fare=0
Mean,32.204208,32.75565
Median,14.454200,14.50000


*Your reasoning here.*
The key idea is that a dataset alone cannot tell you whether 0 means "free ticket" or "unknown fare". That requires domain knowledge or documentation. Since these rows otherwise appear valid, most analysts would keep them rather than convert them to missing values.

## ✏️ Your Turn
Pick one numeric column from `pokemon.csv`. In 2–3 sentences, describe it using **one center and one spread statistic that fit its shape**, and justify your choice using the gap between its mean and median.

In [23]:
# Your turn here
pokemon = pd.read_csv('pokemon.csv')
pokemon['Attack'].mean()
pokemon['Attack'].median()

q1 = pokemon['Attack'].quantile(0.25)
q3 = pokemon['Attack'].quantile(0.75)
iqr = q3 - q1

print("Mean:", pokemon['Attack'].mean())
print("Median:", pokemon['Attack'].median())
print("IQR:", iqr)

Mean: 79.00125
Median: 75.0
IQR: 45.0


*Your description here.*
The Attack stat is best described using the median for center and the IQR (interquartile range) for spread because the distribution is typically right-skewed. A few Pokémon such as legendaries and mega evolutions have extremely high Attack values, which pull the mean above the median. Because there is a noticeable gap between the mean and median, the median provides a more representative "typical" Attack value, while the IQR describes the spread of the middle 50% of Pokémon without being overly influenced by those extreme high-Attack outliers.
If the dataset's mean and median are nearly identical, then the distribution is more symmetric, and mean + standard deviation would be the more appropriate pair of statistics.


## 🔥 Challenge (Optional) — Median Absolute Deviation
The **median absolute deviation (MAD)** is a robust replacement for the standard deviation: the median of the distances between each value and the median.
```python
mad = (s - s.median()).abs().median()
```
Compute the MAD and the std for `fares` and `with_outlier` from section 2. How much does each one change when the outlier is added?

In [24]:
# Your code here
# Original fares
fare_std = fares.std()
fare_mad = (fares - fares.median()).abs().median()

# Fares with an outlier added
outlier_std = with_outlier.std()
outlier_mad = (with_outlier - with_outlier.median()).abs().median()

print("Original fares")
print("STD:", fare_std)
print("MAD:", fare_mad)

print("\nWith outlier")
print("STD:", outlier_std)
print("MAD:", outlier_mad)

print("\nChange")
print("STD change:", outlier_std - fare_std)
print("MAD change:", outlier_mad - fare_mad)

Original fares
STD: 4.69041575982343
MAD: 3.0

With outlier
STD: 198.86117435705407
MAD: 4.5

Change
STD change: 194.17075859723064
MAD change: 1.5


If the outlier makes the standard deviation jump but hardly affects the MAD, that's evidence that MAD is a robust measure of spread while standard deviation is highly sensitive to outliers.

---
✅ Next up → [Shape and Outliers](03-Shape_and_Outliers.ipynb): measuring how lopsided a column is, and finding the values that don't fit.